# 01 — Data and experiment protocol

**Goal:** prepare a reproducible comparison of `HuggingFaceTB/SmolLM2-360M` before and after maths fine-tuning.

This notebook prepares data and freezes the rules of the experiment. It **does not train a model or produce an accuracy score**. Notebook 02 will measure the baseline using this notebook's saved bundle.

**Kaggle settings:** Internet **On**, accelerator **None** (CPU is enough). Choose **Run All**, then **Save Version → Save & Run All** to retain outputs. All sources are public; no Hugging Face token is required.

**GitHub workflow:** import this notebook once from the repository. Each run fetches a fresh checkout of `main`, so shared code/config changes arrive without copying cells. The exact commit is recorded. Importing a notebook alone is not continuous synchronization. If the launcher cells themselves change, re-import the notebook. To reproduce an older experiment, set `MATH_PROJECT_REF` to its saved commit before running, or check out that commit locally.


## 1. Load the current project from GitHub

A fresh temporary checkout avoids overwriting local edits and prevents stale Python imports. The rest of the pipeline runs in a separate process. Source code is kept under `/tmp`; only experiment outputs go into `/kaggle/working`.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import tempfile

REPOSITORY = "https://github.com/sahan24854/Fine-Tuning-LLM-for-math.git"
PROJECT_REF = os.environ.get("MATH_PROJECT_REF", "main")
if PROJECT_REF.startswith("-"):
    raise ValueError("Use a branch, tag, or full commit SHA as MATH_PROJECT_REF")
PROJECT = Path(tempfile.mkdtemp(prefix="math-project-"))
subprocess.run(["git", "init", "--quiet", str(PROJECT)], check=True)
subprocess.run(["git", "-C", str(PROJECT), "remote", "add", "origin", REPOSITORY], check=True)
subprocess.run(["git", "-C", str(PROJECT), "fetch", "--depth", "1", "origin", PROJECT_REF], check=True)
subprocess.run(["git", "-C", str(PROJECT), "checkout", "--detach", "--quiet", "FETCH_HEAD"], check=True)
CODE_COMMIT = subprocess.check_output(["git", "-C", str(PROJECT), "rev-parse", "HEAD"], text=True).strip()
print("Repository:", REPOSITORY)
print("Code commit:", CODE_COMMIT)
print("Project checkout:", PROJECT)


## 2. Install only the data-preparation dependencies

An isolated Python environment avoids replacing Kaggle's PyTorch/CUDA installation. We need the **tokenizer**, not the model's weights, to measure sequence lengths. The top-level versions are pinned in GitHub; the bundle also records all resolved package versions.


In [ ]:
ENVIRONMENT = Path(tempfile.mkdtemp(prefix="math-data-env-"))
subprocess.run([sys.executable, "-m", "venv", str(ENVIRONMENT)], check=True)
PYTHON = ENVIRONMENT / "bin" / "python"
subprocess.run([str(PYTHON), "-m", "pip", "install", "--disable-pip-version-check",
                "-r", str(PROJECT / "requirements-data.txt")], check=True)
subprocess.run([str(PYTHON), "-m", "pip", "check"], check=True)


## 3. Inspect the experiment rules

- **GSM8K:** preserve the official test set. Remove exact training/test overlap and duplicate training questions; reserve 10% of the remaining training pool for validation with seed 42.
- **SVAMP:** keep all 1,000 problems for a separate external evaluation; do not use them for training or model selection.
- **Arithmetic:** generate 500 evaluation-only questions covering four operations and two-step problems, with small and large operands. These are fresh controlled probes, not proof of zero pretraining overlap or generalization to all mathematics.
- **Training:** use solution-only loss; exclude and report overlength training examples instead of cutting off their final answer.
- **Evaluation:** use identical fixed three-shot demonstrations, prompts, and deterministic generation for baseline and adapted model. Report zero-shot separately. Test answers are stored for scoring but are never model input.

`configs/data_protocol.json` in GitHub is the source of truth. Change it there for a new experiment. Preserve completed bundles for comparisons already in progress.


In [ ]:
CONFIG = json.loads((PROJECT / "configs/data_protocol.json").read_text())
print(json.dumps({key: CONFIG[key] for key in ["experiment_name", "model", "seed",
    "validation_fraction", "max_training_tokens", "few_shot_count", "generation", "scoring"]}, indent=2))


## 4. Prepare and audit the data

Downloads are tied to immutable Git commits and verified with SHA-256 checksums. The model's requested revision is resolved to an exact Hugging Face commit before downloading its tokenizer. Every later model load must use that resolved revision.

This cell also measures training lengths, chooses demonstrations **from training only**, and checks that every evaluation prompt fits the reserved context budget. It fails explicitly if the protocol cannot be satisfied. Network errors should be fixed by enabling Internet, not by disabling TLS or checksum verification.


In [ ]:
from datetime import datetime, timezone
from uuid import uuid4

OUTPUT_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd() / "artifacts"
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:8]
BUNDLE = OUTPUT_ROOT / ("data-protocol-" + RUN_ID)
CACHE = Path(tempfile.gettempdir()) / "math-project-source-cache"
subprocess.run([str(PYTHON), str(PROJECT / "scripts/prepare_data.py"),
                "--config", str(PROJECT / "configs/data_protocol.json"),
                "--output", str(BUNDLE), "--cache", str(CACHE)], check=True, cwd=PROJECT)
subprocess.run([str(PYTHON), str(PROJECT / "scripts/verify_bundle.py"), str(BUNDLE)], check=True, cwd=PROJECT)


## 5. Read the audit report

Counts may differ from a simple 90/10 split because exact overlaps, duplicate training questions, and long training examples are excluded and reported. Official test sets remain intact. Exact matching does not detect all paraphrases; public benchmark pretraining contamination remains unknown.

Only aggregate statistics are shown. Avoid browsing test solutions while choosing prompts or hyperparameters.


In [ ]:
AUDIT = json.loads((BUNDLE / "audit.json").read_text())
MANIFEST = json.loads((BUNDLE / "manifest.json").read_text())
from IPython.display import Markdown, display

display(Markdown("| Split | Examples |\n|---|---:|\n" + "\n".join(
    f"| {name} | {count:,} |" for name, count in AUDIT["final_counts"].items())))
print("Training duplicates removed:", AUDIT["training_duplicates_removed"])
print("Training/test exact overlaps removed:", AUDIT["training_test_overlap_removed"])
print("Overlength training examples excluded:", AUDIT["excluded_training_rows"])
print("Training token lengths:", AUDIT["training_tokens_before_filter"])
print("Evaluation prompt lengths:")
print(json.dumps(AUDIT["evaluation_prompt_tokens"], indent=2))
print("Frozen model revision:", MANIFEST["model"]["revision"])
print("Code commit:", MANIFEST["code_commit"])
print("Status:", MANIFEST["status"])


## 6. Save the bundle for the baseline notebook

The bundle includes dataset JSONL files, fixed demonstrations, the resolved configuration, tokenizer files, package versions, an audit report, and a manifest with file hashes. Keep the original bundle unchanged for both baseline and fine-tuned evaluation.

Save this Kaggle notebook version with outputs. Later, attach its output to Notebook 02 through Kaggle's **Add Input** flow (select notebook output), or download the archive. This notebook deliberately does not download model weights or consume GPU training time.


In [ ]:
import shutil
ARCHIVE = Path(shutil.make_archive(str(BUNDLE), "zip", root_dir=BUNDLE))
print("Bundle directory:", BUNDLE)
print("Archive:", ARCHIVE)
print("Files:")
for path in sorted(BUNDLE.iterdir()):
    print(" -", path.name)
print("Data preparation complete. Baseline evaluation and fine-tuning have not run.")
